# 01 ? Prepare paired real-noise images

Build one fixed manifest of 5,000 aligned SIDD and PolyU noisy/clean pairs. The split is 4,000 train, 500 validation, and 500 test. Keep every crop from an original scene in the same split. No synthetic noise is generated.

Obtain the datasets under their own terms and extract the paired subsets under `data/raw/`. This notebook expects SIDD `train_sub512/{GT,LQ}` PNG files and PolyU `train_sub/{GT,LQ}` JPG files. Adjust the two directory settings below if your archive layout differs. Confirm that filenames match and images are spatially aligned; equal dimensions alone do not prove alignment.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().resolve()
if not (ROOT / 'app').is_dir(): ROOT = ROOT.parent
if not (ROOT / 'app').is_dir(): raise FileNotFoundError('Open the project root or notebooks/ in VS Code')
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data'
INVENTORY = DATA / 'inventory'
INVENTORY.mkdir(parents=True, exist_ok=True)
print('Project:', ROOT)

## 1. Locate the two paired subsets

Change these paths to match your extracted data. Both directories must have `GT` for clean images and `LQ` for noisy images. Do not mix the datasets' original holdout/test sets into this training inventory.

In [ ]:
SIDD_BASE = DATA / 'raw/SIDD/SIDD/train_sub512'
POLYU_BASE = DATA / 'raw/PolyU/PolyU-Real-World-Noisy-Images-Dataset/train_sub'
for name, base in [('SIDD', SIDD_BASE), ('PolyU', POLYU_BASE)]:
    for part in ('GT', 'LQ'):
        directory = base / part
        if not directory.is_dir(): raise FileNotFoundError(f'{name} {part}: {directory}')
    print(name, 'clean:', base / 'GT', 'noisy:', base / 'LQ')

## 2. Write curated inventories

Inventory filenames and image paths must match `scripts/prepare_mixed_dataset.py`: `data/inventory/SIDD.csv` and `PolyU.csv`, with paths relative to `data/`. SIDD is grouped by the leading scene ID; PolyU is grouped by the filename before the `_s` patch suffix. Review several pairs and group IDs before creating the split. Existing inventories are left untouched.

In [ ]:
import csv
from PIL import Image

def write_inventory(name, base, suffix):
    output = INVENTORY / f'{name}.csv'
    if output.exists():
        print('Using existing inventory:', output)
        return output
    gt, lq = base / 'GT', base / 'LQ'
    clean_files = sorted(gt.glob(f'*{suffix}'))
    if not clean_files: raise ValueError(f'No {suffix} clean images in {gt}')
    records = []
    for clean in clean_files:
        noisy = lq / clean.name
        if not noisy.is_file(): raise FileNotFoundError(f'Missing noisy mate: {noisy}')
        stem = clean.stem.rsplit('_s', 1)[0]
        group = stem.split('_')[0] if name == 'SIDD' else stem
        if not group: raise ValueError(f'Cannot identify scene for {clean.name}')
        with Image.open(clean) as a, Image.open(noisy) as b:
            if a.size != b.size: raise ValueError(f'Dimension mismatch: {clean.name}')
        records.append({'noisy_path': noisy.relative_to(DATA).as_posix(),
                        'clean_path': clean.relative_to(DATA).as_posix(),
                        'group_id': group})
    with output.open('w', newline='', encoding='utf8') as file:
        writer = csv.DictWriter(file, fieldnames=['noisy_path','clean_path','group_id'])
        writer.writeheader(); writer.writerows(records)
    print(name, len(records), 'pairs,', len({r['group_id'] for r in records}), 'scene groups')
    return output

write_inventory('SIDD', SIDD_BASE, '.png')
write_inventory('PolyU', POLYU_BASE, '.jpg')

## 3. Create and verify the one shared split

The script checks files, pair dimensions, exact source quotas, and group separation. It assigns whole scenes to train/validation/test and samples crops evenly within each split, omitting surplus crops. It fails without writing if the scene groups cannot support the quotas. Review group definitions; never move crops from the same scene into different splits. `splits.csv` is deliberately never overwritten.

In [ ]:
from scripts.prepare_mixed_dataset import prepare
from app.experiments import dataset_counts
manifest = DATA / 'splits.csv'
if manifest.exists():
    print('Using existing manifest:', manifest)
else:
    manifest = prepare(seed=42)
print('Counts:', dataset_counts(manifest))

In [ ]:
import csv
from collections import Counter
from app.data import DenoiseDataset
with manifest.open(newline='', encoding='utf8') as file:
    rows = list(csv.DictReader(file))
print('Source/split:', dict(Counter((r['source'], r['split']) for r in rows)))
for split in ('train', 'val', 'test'):
    dataset = DenoiseDataset(manifest, split, patch=128)
    noisy, clean = dataset[0]
    assert noisy.shape == clean.shape == (3, 128, 128)
    print(split, len(dataset), 'first pair:', rows[next(i for i,r in enumerate(rows) if r['split']==split)]['noisy_path'])
print('Ready for notebook 02. Keep this manifest fixed for all models.')